# Chapter 6: individuals chart, CUSUM and EWMA on the forty Kestrel lager fills

Limits come from the first 20 fills only. Look for: no point outside the limits, the eight in a row rule at fill 40,
CUSUM crossing at fill 34 and EWMA at fill 35. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
b = load("Batch")
base = b[b.IsBaseline]
centre = base.FinalGravity.mean()
sigma = base.MovingRange.mean() / 1.128
ucl, lcl = centre + 3 * sigma, centre - 3 * sigma
print(f"centre {centre:.3f}  sigma {sigma:.3f}  limits {lcl:.2f} to {ucl:.2f}")

In [ ]:
v = b.FinalGravity.to_numpy()
side = np.sign(v - centre)
run8 = next((i + 1 for i in range(7, len(v)) if abs(side[i - 7:i + 1].sum()) == 8), None)
k, h = 0.064, 0.64          # half a sigma and five sigma, as printed in 6.4
cusum, c = [], 0.0
for x in v:
    c = max(0.0, c + x - (centre + k)); cusum.append(c)
ewma, z = [], centre
for x in v:
    z = 0.2 * x + 0.8 * z; ewma.append(z)
half = 3 * 0.129 * (0.2 / 1.8) ** 0.5
print("outside limits:", int(((v > ucl) | (v < lcl)).sum()))
print("eight in a row fires at fill", run8)
print("CUSUM crosses", round(h, 2), "at fill", next(i + 1 for i, x in enumerate(cusum) if x >= h))
print("EWMA leaves +/-", round(half, 3), "at fill", next(i + 1 for i, x in enumerate(ewma) if abs(x - centre) > half))

In [ ]:
x = b.FillSequence
fig, (a, c2, e) = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
a.plot(x, v, "o-", color="0.3", ms=3)
for y, s in ((centre, "-"), (ucl, "--"), (lcl, "--")):
    a.axhline(y, color="0.6", ls=s)
a.axvline(20.5, color="0.85"); a.set_ylabel("Final gravity (degrees Plato)")
a.set_title("Final gravity by fill, Kestrel Lager, degrees Plato", loc="left")
c2.plot(x, cusum, color="tab:blue"); c2.axhline(h, color="0.6", ls="--"); c2.set_ylabel("CUSUM")
e.plot(x, ewma, color="tab:blue"); e.axhline(centre + half, color="0.6", ls="--"); e.axhline(centre - half, color="0.6", ls="--")
e.set_ylabel("EWMA"); e.set_xlabel("Fill sequence"); plt.tight_layout(); plt.show()

## Capability on the baseline and after the fix

In [ ]:
s_all = base.FinalGravity.std(ddof=1)
near = min(2.6 - centre, centre - 1.8)
print(f"Cp {0.8 / (6 * sigma):.2f}  Cpk {near / (3 * sigma):.2f} (1.02 with sigma rounded to 0.129)")
print(f"Pp {0.8 / (6 * s_all):.2f}  Ppk {near / (3 * s_all):.2f}")
print(f"last ten fills average {v[30:].mean():.3f}")
after = load("BatchAfterFix").FinalGravity
print(f"after the fix: mean {after.mean():.2f}, sd {after.std(ddof=1):.3f}, Ppk {(2.6 - after.mean()) / (3 * after.std(ddof=1)):.2f}")